## Solucion: 02_sql_discovery
### Ing. Jorge Uriel liévano Cifuentes
### viernes 19 de junio de 2026
### Bucaramanga - Colombia - UDES

In [1]:
# PASO 1 — Importar librerias
import sqlite3
import pandas as pd

In [2]:
# PASO 2 — Cargar el archivo entas_techventas.csv
df = pd.read_csv(
    '../data/ventas_techventas.csv',
    sep=',',
    quotechar='"',    # Carácter que delimita campos de texto
    doublequote=True, # Maneja comillas dobles dentro del texto    
    encoding='latin1',
    parse_dates=['fecha'],
    engine='python',  # Más lento pero más tolerante
    # Esta columna la interprete como fecha.
)

In [3]:
# PASO 3 — Crear base SQLite en memoria
conn = sqlite3.connect(":memory:") # Guarda la conexión para reutilizarla en todas las consultas.

# Crear cursor para ejecutar consultas SQL
cursor = conn.cursor()  # Creamos el cursor que enviará instrucciones SQL a la base de datos.

print(conn)

In [4]:
# PASO 4 — Importar DataFrame a SQLite
df.to_sql(
    name="ventas",
    con=conn,
    if_exists="replace",
    index=False
)

60

In [5]:
# PASO 5 — Verificar tabla creada
pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

,name
0,ventas


## Crear tabla vendedores (Q5)

In [6]:
# PASO 6 — Crear tabla
conn.execute("""
CREATE TABLE vendedores(
    vendedor TEXT,
    zona TEXT,
    meta_mensual REAL
)
""")

In [7]:
# PASO 7 — Insertar datos
conn.executemany(
"""
INSERT INTO vendedores
VALUES (?,?,?)
""",
[
    ("Ana López","Norte",8000),
    ("Carlos Ruiz","Sur",7500),
    ("Luis Mora","Norte",8000),
    ("Maria Torres","Centro",7000)
]
)

In [8]:
# PASO 8 — Confirmar cambios
conn.commit()

In [9]:
# PASO 9 — Verificar vendedores
pd.read_sql(
    "SELECT * FROM vendedores",
    conn
)

,vendedor,zona,meta_mensual
0,Ana López,Norte,8000.0
1,Carlos Ruiz,Sur,7500.0
2,Luis Mora,Norte,8000.0
3,Maria Torres,Centro,7000.0


## Q1 — SELECT DISTINCT
Lista todos los productos únicos disponibles usando un alias de columna descriptivo.

In [10]:
# PASO 10 — Verificar vendedores
q1 = """
SELECT DISTINCT
       producto AS producto_disponible
FROM ventas
WHERE producto IS NOT NULL
"""

pd.read_sql(q1, conn)

,producto_disponible
0,Laptop Pro 15
1,Mouse Inalambrico
2,Teclado Mecanico
3,SSD 1TB
4,Router WiFi 6


## Q2 — WHERE + BETWEEN
Filtra pedidos del primer trimestre (ene–mar 2024) con cantidad mayor a 5 unidades.

In [11]:
q2 = """
SELECT *
FROM ventas
WHERE fecha BETWEEN '2024-01-01'
                AND '2024-03-31'
  AND cantidad > 5;
"""

pd.read_sql(q2, conn)

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1002,2024-01-07 00:00:00,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz
1,1004,2024-01-12 00:00:00,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora
2,1006,2024-01-18 00:00:00,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20.0,95.0,0.00,Maria Torres
3,1008,2024-01-22 00:00:00,C006,NetPrime,Norte,Router WiFi 6,Redes,8.0,120.0,0.00,Luis Mora
4,1009,2024-01-25 00:00:00,C003,DataSolutions,Centro,SSD 1TB,Almacenamiento,12.0,95.0,0.10,Maria Torres
5,1012,2024-02-05 00:00:00,C008,BetaSoft,Centro,Mouse Inalambrico,Perifericos,30.0,25.0,0.05,Maria Torres
6,1015,2024-02-12 00:00:00,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,25.0,95.0,0.10,Maria Torres
7,1018,2024-02-20 00:00:00,C002,Innovatek,Sur,SSD 1TB,Almacenamiento,18.0,95.0,0.00,Carlos Ruiz
8,1019,2024-02-22 00:00:00,C007,AlphaTech,Sur,Mouse Inalambrico,Perifericos,12.0,25.0,0.00,Maria Torres
9,1022,2024-03-04 00:00:00,C001,TechCorp SA,Norte,SSD 1TB,Almacenamiento,30.0,95.0,0.15,Carlos Ruiz


## Q3 — GROUP BY + HAVING
Vendedores cuyo ingreso bruto total (cantidad × precio) supera $10,000 USD.

In [12]:
q3 = """
SELECT
    vendedor,
    SUM(cantidad * precio_unitario) AS ingreso_bruto
FROM ventas
GROUP BY vendedor
HAVING ingreso_bruto > 10000;
"""

pd.read_sql(q3, conn)

,vendedor,ingreso_bruto
0,Ana Lï¿½ï¿,20810.0
1,Carlos Ruiz,21355.0
2,Luis Mora,19830.0
3,Maria Torres,11860.0


## Q4 — COUNT + SUM + AVG
Por categoría: total de pedidos, suma de unidades vendidas y precio unitario promedio.

In [13]:
q4 = """
SELECT
    categoria,
    COUNT(*) AS total_pedidos,
    SUM(cantidad) AS total_unidades,
    AVG(precio_unitario) AS precio_promedio
FROM ventas
GROUP BY categoria;
"""

pd.read_sql(q4, conn)

,categoria,total_pedidos,total_unidades,precio_promedio
0,NaN,11,NaN,NaN
1,Almacenamiento,12,260.0,95.000000
2,Electronica,14,31.0,1139.285714
3,Perifericos,15,215.0,53.000000
4,Redes,8,39.0,120.000000


## Q5 — JOIN
Crea la tabla vendedores (abajo) y calcula el % de cumplimiento de meta de cada uno.

In [14]:
q5 = """
SELECT
    v.vendedor,
    v.zona,
    v.meta_mensual,

    SUM(
        vt.cantidad *
        vt.precio_unitario
    ) AS venta_real,

    ROUND(
        SUM(
            vt.cantidad *
            vt.precio_unitario
        ) * 100.0 /
        v.meta_mensual,
        2
    ) AS porcentaje_cumplimiento

FROM ventas vt
JOIN vendedores v
     ON vt.vendedor = v.vendedor

GROUP BY
    v.vendedor,
    v.zona,
    v.meta_mensual;
"""

pd.read_sql(q5, conn)

,vendedor,zona,meta_mensual,venta_real,porcentaje_cumplimiento
0,Carlos Ruiz,Sur,7500.0,21355.0,284.73
1,Luis Mora,Norte,8000.0,19830.0,247.88
2,Maria Torres,Centro,7000.0,11860.0,169.43


## Q6 — Subconsulta
Encuentra el cliente que generó el mayor ingreso total en el primer semestre.

In [15]:
q6 = """
SELECT *
FROM
(
    SELECT
        cliente_id,
        SUM(
            cantidad *
            precio_unitario
        ) AS ingreso_total
    FROM ventas
    GROUP BY cliente_id
)
ORDER BY ingreso_total DESC
LIMIT 1;
"""

pd.read_sql(q6, conn)

,cliente_id,ingreso_total
0,C001,19025.0


In [16]:
# Cierre del notebook
conn.close()